<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/08_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.7.0?labpath=examples%2F08_validation.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/08_validation.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>

# 08 — Validation, Schema & Cost Estimation

Demonstrates the validation layer:
- `DataSchema` — define, validate, infer, save, load
- `estimate_cost` — gate count and NISQ safety
- `PipelineResult.cost` and `.audit_log`
- `result.summary()`

```bash
pip install quprep
```

In [ ]:
import json
import warnings

import numpy as np

import quprep as qd
from quprep.core.dataset import Dataset
from quprep.validation import QuPrepWarning, validate_dataset

rng = np.random.default_rng(42)
X = rng.uniform(0.0, 1.0, size=(20, 6)).astype(np.float64)
ds_clean = Dataset(data=X, feature_names=[f"f{i}" for i in range(6)])

print(f"quprep {qd.__version__}")

## 1. NaN warning from validate_dataset

In [ ]:
X_nan = X.copy()
X_nan[0, 0] = np.nan
X_nan[2, 0] = np.nan
ds_nan = Dataset(data=X_nan, feature_names=[f"f{i}" for i in range(6)])

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    validate_dataset(ds_nan, context="notebook")

for w in caught:
    if issubclass(w.category, QuPrepWarning):
        print(f"Warning: {w.message}")

## 2. DataSchema — define and validate

In [ ]:
schema = qd.DataSchema([
    qd.FeatureSpec(f"f{i}", dtype="continuous", min_value=0.0, max_value=1.0)
    for i in range(6)
])

schema.validate(ds_clean)
print("Clean dataset: OK")

X_bad = X.copy()
X_bad[0, 2] = 1.5  # exceeds max_value=1.0
ds_bad = Dataset(data=X_bad, feature_names=[f"f{i}" for i in range(6)])

try:
    schema.validate(ds_bad)
except qd.SchemaViolationError as e:
    print(f"Violation: {str(e).splitlines()[0]}")

## 3. Infer schema from data

In [ ]:
inferred = qd.DataSchema.infer(ds_clean)
print(f"Features inferred : {len(inferred.features)}")
f0 = inferred.features[0]
print(f"f0 range          : [{f0.min_value:.3f}, {f0.max_value:.3f}]")
inferred.validate(ds_clean)
print("Inferred schema validates source: OK")

## 4. Save and load schema (JSON)

In [ ]:
schema_path = "/tmp/quprep_schema.json"
with open(schema_path, "w") as f:
    f.write(inferred.to_json())

with open(schema_path) as f:
    restored = qd.DataSchema.from_json(f.read())

restored.validate(ds_clean)
print("Loaded + validated: OK")
print(f"First entry: {json.loads(inferred.to_json())[0]}")

## 5. Cost estimation

In [ ]:
import pandas as pd

rows = []
for encoder_cls, label in [
    (qd.AngleEncoder,          "AngleEncoder"),
    (qd.AmplitudeEncoder,      "AmplitudeEncoder"),
    (qd.IQPEncoder,            "IQPEncoder"),
    (qd.EntangledAngleEncoder, "EntangledAngleEncoder"),
]:
    cost = qd.estimate_cost(encoder_cls(), n_features=6)
    rows.append({
        "encoder":     label,
        "n_qubits":    cost.n_qubits,
        "depth":       cost.circuit_depth,
        "gates":       cost.gate_count,
        "2q_gates":    cost.two_qubit_gates,
        "nisq_safe":   cost.nisq_safe,
    })

pd.DataFrame(rows)

## 6. Pipeline with schema — cost and audit log

In [ ]:
pipeline = qd.Pipeline(
    cleaner=qd.Imputer(),
    reducer=qd.PCAReducer(n_components=3),
    encoder=qd.AngleEncoder(),
    schema=inferred,
)
result = pipeline.fit_transform(ds_clean)

print(f"encoding  : {result.cost.encoding}")
print(f"n_qubits  : {result.cost.n_qubits}")
print(f"nisq_safe : {result.cost.nisq_safe}")
print()

audit = pd.DataFrame(result.audit_log)
audit

## 7. PipelineResult.summary()

In [ ]:
print(result.summary())